# Problem

建立一個以 MNIST 為訓練資料的全連接神經網路，並提供本機 Gradio 手寫數字展示介面。評估流程必須保留獨立測試資料，避免把測試結果當成訓練期間的調整依據。


In [ ]:
# 1. 安裝所需套件
!pip install gradio

# 2. 載入模型開發與影像處理套件
import numpy as np
import cv2
import tensorflow as tf
from tensorflow.keras.datasets import mnist
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization, Activation
from tensorflow.keras.utils import to_categorical
from PIL import Image
import gradio as gr

# 讓模型初始化與資料切分可重現
tf.keras.utils.set_random_seed(2026)

# 3. 準備並處理 MNIST 訓練資料
(x_train, y_train), (x_test, y_test) = mnist.load_data()

# 依照 DNN 的要求，拉平影像並做 0~1 的正規化
x_train = x_train.reshape(60000, 784) / 255.0
x_test = x_test.reshape(10000, 784) / 255.0

# 輸出轉為 One-hot encoding (10種分類)
y_train = to_categorical(y_train, 10)
y_test = to_categorical(y_test, 10)

print("====== DNN：資料準備完成！======")


# Method

- 將 MNIST 影像拉平成 784 維向量並正規化至 0–1。
- 使用含 Batch Normalization 與 Dropout 的全連接神經網路。
- 固定隨機種子為 2026，並以訓練資料的 10% 作為 validation split。
- 訓練完成後才使用保留的 `x_test`、`y_test` 做一次最終評估。
- Gradio 輸入先以邊界框裁切、縮放並置中；這是展示用的啟發式前處理。


In [ ]:
# 建立 DNN 模型
model = Sequential()

# 第 1 層隱藏層 (512個神經元 + 批次標準化 + Dropout)
model.add(Dense(512, input_dim=784, use_bias=False))
model.add(BatchNormalization())
model.add(Activation('relu'))
model.add(Dropout(0.2))

# 第 2 層隱藏層 (512個神經元)
model.add(Dense(512, use_bias=False))
model.add(BatchNormalization())
model.add(Activation('relu'))
model.add(Dropout(0.2))

# 第 3 層隱藏層 (256個神經元)
model.add(Dense(256, use_bias=False))
model.add(BatchNormalization())
model.add(Activation('relu'))

# 輸出層 (10種分類的機率)
model.add(Dense(10, activation='softmax'))

# 使用更好的 Adam 優化器與分類專用 Loss
model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

# 自動顯示網路與開始訓練
model.summary()

print("開始訓練深層 DNN 模型...")
history = model.fit(
    x_train,
    y_train,
    batch_size=128,
    epochs=15,
    validation_split=0.1,
)
print("模型訓練完成。")

# 訓練與模型選擇完成後，只評估保留的 test split 一次
test_loss, test_accuracy = model.evaluate(x_test, y_test, verbose=0)


# Results

執行 Notebook 時，`history` 會保存訓練與 validation 指標，模型也會對保留的 test split 做一次最終評估。公開版本不保存執行輸出，因此不宣稱特定準確率；結果應在實際執行環境中重新產生。


# Limitations

- DNN 不具備卷積模型的平移不變性，手寫位置與筆畫粗細仍可能影響預測。
- 邊界框置中是啟發式方法，不能保證涵蓋所有輸入樣式。
- `validation_split=0.1` 取自訓練資料；test split 僅用於訓練後評估。
- Gradio 僅以 `share=False`、`debug=False` 在本機啟動，公開版本不建立外部分享連結。


In [ ]:
def resize_image_dnn_advanced(inp):
    # 1. 將 Gradio 圖層轉換成背景為白色的影像
    image = np.array(inp["layers"][0], dtype=np.float32).astype(np.uint8)
    image_pil = Image.fromarray(image)
    background = Image.new("RGB", image_pil.size, (255, 255, 255))
    background.paste(image_pil, mask=image_pil.split()[3])

    # 轉換為黑底白字
    image_gray = background.convert("L")
    img_array = 255 - np.array(image_gray)

    # 2. 自動鎖定「畫筆邊界框 (Bounding Box)」
    coords = cv2.findNonZero(img_array)
    if coords is not None:
        # 切下字跡的部分
        x, y, w, h = cv2.boundingRect(coords)
        cropped_digit = img_array[y:y+h, x:x+w]

        # 3. 把字跡長邊縮放成 20 像素 (還原 MNIST 訓練集的預設做法)
        if w > 0 and h > 0:
            if w > h:
                new_w = 20
                new_h = max(1, int(h * (20.0 / w)))
            else:
                new_h = 20
                new_w = max(1, int(w * (20.0 / h)))
            resized_digit = cv2.resize(cropped_digit, (new_w, new_h), interpolation=cv2.INTER_LANCZOS4)
        else:
            resized_digit = cropped_digit
            new_w, new_h = w, h

        # 4. 把縮放後的字跡貼到 28x28 全黑畫布的「正中央」
        final_img = np.zeros((28, 28), dtype=np.uint8)
        off_x = (28 - new_w) // 2
        off_y = (28 - new_h) // 2
        final_img[off_y:off_y+new_h, off_x:off_x+new_w] = resized_digit
    else:
        final_img = np.zeros((28, 28), dtype=np.uint8)

    # 5. 拉平成 784，符合 DNN 需要的 1D 格式
    final_input = final_img.reshape(1, 784) / 255.0
    return final_input

def recognize_digit_dnn(inp):
    try:
        # 將畫板影像送入處理常式
        img_array = resize_image_dnn_advanced(inp)
        # 用升級版的 DNN 預測
        prediction = model.predict(img_array).flatten()
        labels = list('0123456789')
        return {labels[i]: float(prediction[i]) for i in range(10)}
    except Exception as e:
        return {"發生錯誤，請重畫": 0.0}

# 啟動 Gradio (拿掉不支援的畫筆粗細參數，恢復原廠設定)
iface = gr.Interface(
    fn=recognize_digit_dnn,
    inputs=gr.Sketchpad(),
    outputs=gr.Label(num_top_classes=3),
    title="😎",
    description="請在畫板輸入一個數字；前處理會裁切非空白區域、縮放並置中，結果仍會受筆畫與位置影響。"
)

iface.launch(share=False, debug=False)
